Data Preparation & Feature Engineering

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.spatial import KDTree
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.preprocessing import LabelEncoder
import joblib

# Load Data
fraud_tx = pd.read_csv('fraud_transactions.csv')
withdrawals = pd.read_csv('atm_withdrawals.csv')
atms = pd.read_csv('atms.csv')
accounts = pd.read_csv('accounts.csv')

# Build Master Timeline
fraud_tx['event_type'] = 'digital_transfer'
withdrawals['event_type'] = 'atm_cashout'
withdrawals = withdrawals.rename(columns={'account_id': 'destination_account'})
master_timeline = pd.concat([fraud_tx, withdrawals], axis=0, ignore_index=True)
master_timeline = master_timeline.sort_values(by=['case_id', 'timestamp_min']).reset_index(drop=True)

# Merge Geographic Clues
master_timeline = pd.merge(
    master_timeline, 
    accounts[['account_id', 'bank_id']], 
    left_on='destination_account', 
    right_on='account_id', 
    how='left'
)

master_timeline[['city', 'state', 'bank_id']] = master_timeline[['city', 'state', 'bank_id']].fillna('UNKNOWN')
master_timeline['amount_inr'] = master_timeline['amount_inr'].fillna(0.0)

# Encode Categories
city_enc, state_enc, bank_enc = LabelEncoder(), LabelEncoder(), LabelEncoder()
master_timeline['city_idx'] = city_enc.fit_transform(master_timeline['city'])
master_timeline['state_idx'] = state_enc.fit_transform(master_timeline['state'])
master_timeline['bank_idx'] = bank_enc.fit_transform(master_timeline['bank_id'])

num_cities, num_states, num_banks = len(city_enc.classes_), len(state_enc.classes_), len(bank_enc.classes_)

# Extract Sequences
X_seq_list, y_coord_list = [], []
atm_coords_dict = atms.set_index('atm_id')[['latitude', 'longitude']].to_dict('index')

for case, group in master_timeline.groupby('case_id'):
    digital_hops = group[group['event_type'] == 'digital_transfer']
    cashout = group[group['event_type'] == 'atm_cashout']
    
    if not digital_hops.empty and not cashout.empty:
        target_atm = cashout['atm_id'].iloc[0]
        if target_atm in atm_coords_dict:
            hops_features = []
            for _, row in digital_hops.iterrows():
                amt = np.log1p(float(row['amount_inr']))
                hour = ((float(row['timestamp_min']) // 60) % 24) / 24.0
                c, s, b = row['city_idx'] / num_cities, row['state_idx'] / num_states, row['bank_idx'] / num_banks
                hops_features.append([amt, hour, c, s, b])
            
            X_seq_list.append(torch.tensor(hops_features, dtype=torch.float32))
            y_coord_list.append([atm_coords_dict[target_atm]['latitude'], atm_coords_dict[target_atm]['longitude']])

# Pad and Standardize
y_arr = np.array(y_coord_list)
lat_mean, lat_std = y_arr[:, 0].mean(), y_arr[:, 0].std()
lon_mean, lon_std = y_arr[:, 1].mean(), y_arr[:, 1].std()

y_norm = np.column_stack([(y_arr[:, 0] - lat_mean) / lat_std, (y_arr[:, 1] - lon_mean) / lon_std])
y_tensor = torch.tensor(y_norm, dtype=torch.float32)
X_padded = pad_sequence(X_seq_list, batch_first=True, padding_value=0.0)

# KD-Tree Setup
all_atm_norm = np.column_stack([
    (atms['latitude'].values - lat_mean) / lat_std,
    (atms['longitude'].values - lon_mean) / lon_std
])
atm_tree = KDTree(all_atm_norm)

Model Architecture & Training

In [2]:
# DataLoaders
dataset = TensorDataset(X_padded, y_tensor)
train_size = int(0.8 * len(dataset))
train_set, test_set = torch.utils.data.random_split(dataset, [train_size, len(dataset) - train_size])
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

# LSTM Class
class ATMTrajectoryLSTM(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=128):
        super(ATMTrajectoryLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 2))
        
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

# Training Engine
model = ATMTrajectoryLSTM(input_dim=5) 
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

for epoch in range(20):
    model.train()
    running_loss = 0.0
    for bx, by in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * bx.size(0)
    print(f"Epoch {epoch+1}/20 - Loss: {running_loss / len(train_set):.4f}")

Epoch 1/20 - Loss: 0.6872
Epoch 2/20 - Loss: 0.2996
Epoch 3/20 - Loss: 0.1829
Epoch 4/20 - Loss: 0.1054
Epoch 5/20 - Loss: 0.0650
Epoch 6/20 - Loss: 0.1201
Epoch 7/20 - Loss: 0.0516
Epoch 8/20 - Loss: 0.0149
Epoch 9/20 - Loss: 0.0070
Epoch 10/20 - Loss: 0.1018
Epoch 11/20 - Loss: 0.0272
Epoch 12/20 - Loss: 0.0114
Epoch 13/20 - Loss: 0.0624
Epoch 14/20 - Loss: 0.0032
Epoch 15/20 - Loss: 0.0016
Epoch 16/20 - Loss: 0.0018
Epoch 17/20 - Loss: 0.0009
Epoch 18/20 - Loss: 0.0030
Epoch 19/20 - Loss: 0.0037
Epoch 20/20 - Loss: 0.0021


Save Artifacts for FastAPI

In [5]:
# Export variables for backend integration
torch.save(model.state_dict(), 'atm_trajectory_lstm2.pth')
joblib.dump({
    'atm_tree': atm_tree, 'lat_mean': lat_mean, 'lat_std': lat_std,
    'lon_mean': lon_mean, 'lon_std': lon_std, 'city_classes': city_enc.classes_,
    'state_classes': state_enc.classes_, 'bank_classes': bank_enc.classes_
}, 'spatial_artifacts.pkl')

['spatial_artifacts.pkl']

The Inference Test

In [6]:
import torch
import numpy as np
import pandas as pd
import joblib

# 1. Load the raw ATM dataset to map indices back to text IDs
atms = pd.read_csv('atms.csv')

# 2. Load the saved spatial dictionary and KD-Tree
artifacts = joblib.load('spatial_artifacts.pkl')
city_classes = artifacts['city_classes']
state_classes = artifacts['state_classes']
bank_classes = artifacts['bank_classes']
atm_tree = artifacts['atm_tree']
lat_mean, lat_std = artifacts['lat_mean'], artifacts['lat_std']
lon_mean, lon_std = artifacts['lon_mean'], artifacts['lon_std']

# 3. Load the saved neural network weights
# Note: ATMTrajectoryLSTM class must be defined in the runtime before loading
test_model = ATMTrajectoryLSTM(input_dim=5)
test_model.load_state_dict(torch.load('atm_trajectory_lstm.pth'))
test_model.eval() # Lock the weights for testing

# Helper function to safely handle unseen text
def get_encoded_index(text_value, classes_array):
    if text_value in classes_array:
        return np.where(classes_array == text_value)[0][0]
    return 0 # Default to 0 if the detective enters a brand new city/bank

# 4. The Prediction Function
def trigger_alert(digital_trail):
    hops_features = []
    
    for hop in digital_trail:
        amt = np.log1p(hop['amount'])
        
        hours, minutes = map(int, hop['time'].split(':'))
        hour_scaled = (((hours * 60) + minutes) // 60 % 24) / 24.0
        
        c_idx = get_encoded_index(hop['city'], city_classes)
        s_idx = get_encoded_index(hop['state'], state_classes)
        b_idx = get_encoded_index(hop['bank_id'], bank_classes)
        
        city_scaled = c_idx / len(city_classes)
        state_scaled = s_idx / len(state_classes)
        bank_scaled = b_idx / len(bank_classes)
        
        hops_features.append([amt, hour_scaled, city_scaled, state_scaled, bank_scaled])
        
    input_tensor = torch.tensor([hops_features], dtype=torch.float32)
    
    with torch.no_grad():
        pred_norm = test_model(input_tensor).numpy()
        
    distances, nearest_indices = atm_tree.query(pred_norm, k=3)
    
    predicted_lat = (pred_norm[0][0] * lat_std) + lat_mean
    predicted_lon = (pred_norm[0][1] * lon_std) + lon_mean
    
    print(f"Network's Mathematical Prediction: ({predicted_lat:.4f}, {predicted_lon:.4f})")
    print("\n[ALERT] Top 3 Suspected Physical ATMs:")
    for idx in nearest_indices[0]:
        atm_id = atms.iloc[idx]['atm_id'] 
        print(f"-> {atm_id}")

# 5. Run a Mock Cybercrime Case
mock_case = [
    {'amount': 2500, 'time': '09:30', 'city': 'Pune', 'state': 'Maharashtra', 'bank_id': 'BANK_03'},
    {'amount': 45000, 'time': '12:15', 'city': 'Mumbai', 'state': 'Maharashtra', 'bank_id': 'BANK_08'}
]

trigger_alert(mock_case)

Network's Mathematical Prediction: (20.4537, 76.0255)

[ALERT] Top 3 Suspected Physical ATMs:
-> ATM_001725
-> ATM_000769
-> ATM_000611
